In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.types import *

data1 = [(1, 'Joe', 85000, 1), (2, 'Henry', 80000, 2), (3, 'Sam', 60000, 2), (4, 'Max', 90000, 1),
         (5, 'Janet', 69000, 1), (6, 'Randy', 85000, 1), (7, 'Will', 70000, 1)
         ]

data2 = [(1, 'IT'), (2, 'Sales')]

schema1 = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('salary', IntegerType(), True),
    StructField('deptId', IntegerType(), True)
])

schema2 = StructType([
    StructField('departmentId', IntegerType(), False),
    StructField('departmentname', StringType(), True)
])

emp_df = spark.createDataFrame(data=data1,schema=schema1)
dept_df = spark.createDataFrame(data=data2,schema=schema2)

res = emp_df.join(dept_df,emp_df['deptId']==dept_df['departmentId'],how='inner')

w = Window.partitionBy('departmentId').orderBy(f.col('salary').desc())

res = res.withColumn('rank', f.dense_rank().over(w))

res = res.where(f.col('rank')<=3).select('departmentname','name','salary')
res = res.withColumnRenamed('name','employee').withColumnRenamed('salary','Salary')

res.write.format('delta').mode('append').saveAsTable('topThreeEmployees')


In [0]:
%sql
select * from topThreeEmployees